# 02 Train Emulator (PINN from scratch)

Addestramento PINN indipendente dalla MLP: usa solo il dataset condiviso.


In [1]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd / 'PINN', _cwd.parent, _cwd.parent / 'PINN', _cwd.parent.parent]
_PROJECT_ROOT = next((p for p in _candidates if (p / 'src' / 'solsys_emulator').exists()), None)
if _PROJECT_ROOT is None:
    raise RuntimeError('Impossibile trovare la project root con src/solsys_emulator')

_SRC = _PROJECT_ROOT / 'src'
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

print('Project root:', _PROJECT_ROOT)
print('Python executable:', sys.executable)



Project root: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/PINN
Python executable: /Applications/Xcode.app/Contents/Developer/usr/bin/python3


In [2]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch

from solsys_emulator.config import DEFAULT_CHECKPOINT_PATH, DEFAULT_DATASET_PATH
from solsys_emulator.de440_dataset import load_dataset
from solsys_emulator.model import ModelConfig
from solsys_emulator.train import TrainConfig, train_emulator

RUN_STAGE2 = True

dataset = load_dataset(DEFAULT_DATASET_PATH)
print('Dataset source:', dataset.get('metadata', {}).get('sample_source'))
print('States shape:', dataset['states'].shape)
print('Num samples:', len(dataset['times_seconds']))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pinn_ckpt = Path(DEFAULT_CHECKPOINT_PATH)
stage1_ckpt = pinn_ckpt.with_name('emulator_pinn_stage1.pt')
stage2_ckpt = pinn_ckpt.with_name('emulator_pinn_stage2.pt')

model_cfg = ModelConfig(
    num_bodies=len(dataset['bodies']),
    state_mode='position_only',
    hidden_dim=384,
    num_layers=6,
    fourier_features=24,
    min_frequency=1.0,
    max_frequency=6.0,
    frequency_spacing='linear',
    head_layers=1,
    head_hidden_dim=128,
    dropout=0.0,
)

# Stage 1: data-only training (strong baseline)
stage1_cfg = TrainConfig(
    epochs=1200,
    batch_size=384,
    lr=2e-4,
    weight_decay=1e-6,
    val_fraction=0.10,
    split_mode='random',
    shuffle=True,
    early_stopping_patience=220,
    lr_scheduler='cosine',
    min_lr=1e-7,
    nbody_loss_weight=0.0,
    physics_loss_weight=0.0,
    smoothness_loss_weight=0.0,
    position_loss_weight=1.0,
    velocity_loss_weight=1.0,
    grad_clip_norm=1.0,
    selection_metric='val_pos_rmse_km',
    force_chronological_for_derivatives=False,
    sort_train_for_derivatives=False,
    show_progress=True,
    device=device,
)

stage1 = train_emulator(
    dataset,
    train_config=stage1_cfg,
    model_config=model_cfg,
    checkpoint_path=stage1_ckpt,
)
stage1_best_epoch = int(np.argmin(stage1['history']['val_pos_rmse_km'])) + 1
stage1_best_rmse = float(np.min(stage1['history']['val_pos_rmse_km']))

print('Stage1 checkpoint:', stage1_ckpt)
print('Stage1 best epoch:', stage1_best_epoch)
print('Stage1 best val position RMSE [km]:', f'{stage1_best_rmse:,.2f}')

stage2 = None
stage2_best_epoch = None
stage2_best_rmse = None
if RUN_STAGE2:
    # Stage 2: very light physics fine-tuning (keep close to data fit)
    stage2_cfg = TrainConfig(
        epochs=220,
        batch_size=384,
        lr=2e-6,
        weight_decay=1e-6,
        val_fraction=0.10,
        split_mode='random',
        shuffle=False,
        early_stopping_patience=45,
        lr_scheduler='cosine',
        min_lr=5e-7,
        nbody_loss_weight=1e-6,
        nbody_start_epoch=30,
        nbody_warmup_epochs=140,
        nbody_softening_km=80_000.0,
        nbody_relative_floor_km_s2=5e-4,
        physics_loss_weight=0.0,
        smoothness_loss_weight=0.0,
        position_loss_weight=1.0,
        velocity_loss_weight=1.0,
        grad_clip_norm=1.0,
        selection_metric='val_pos_rmse_km',
        force_chronological_for_derivatives=False,
        sort_train_for_derivatives=False,
        show_progress=True,
        device=device,
    )
    stage2 = train_emulator(
        dataset,
        train_config=stage2_cfg,
        model_config=model_cfg,
        checkpoint_path=stage2_ckpt,
        initial_checkpoint_path=stage1_ckpt,
    )
    stage2_best_epoch = int(np.argmin(stage2['history']['val_pos_rmse_km'])) + 1
    stage2_best_rmse = float(np.min(stage2['history']['val_pos_rmse_km']))

    print('Stage2 checkpoint:', stage2_ckpt)
    print('Stage2 best epoch:', stage2_best_epoch)
    print('Stage2 best val position RMSE [km]:', f'{stage2_best_rmse:,.2f}')

# Select best stage by validation physical RMSE (position)
selected_stage = 'stage1'
selected_ckpt = stage1_ckpt
selected_rmse = stage1_best_rmse
if stage2 is not None and stage2_best_rmse is not None and stage2_best_rmse < stage1_best_rmse:
    selected_stage = 'stage2'
    selected_ckpt = stage2_ckpt
    selected_rmse = stage2_best_rmse

shutil.copy2(selected_ckpt, pinn_ckpt)
print('Final PINN checkpoint:', pinn_ckpt)
print('Selected stage:', selected_stage)
print('Best val position RMSE [km]:', f'{selected_rmse:,.2f}')

plt.figure(figsize=(10, 4))
plt.plot(stage1['history']['train_loss'], label='stage1 train(data)')
plt.plot(stage1['history']['val_loss'], label='stage1 val(data)')
if stage2 is not None:
    offset = len(stage1['history']['train_loss'])
    plt.plot(range(offset, offset + len(stage2['history']['train_loss'])), stage2['history']['train_loss'], label='stage2 train(data)')
    plt.plot(range(offset, offset + len(stage2['history']['val_loss'])), stage2['history']['val_loss'], label='stage2 val(data)')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('PINN training history')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(stage1['history']['val_pos_rmse_km'], label='stage1 val position RMSE [km]')
if stage2 is not None:
    plt.plot(range(len(stage1['history']['val_pos_rmse_km']), len(stage1['history']['val_pos_rmse_km']) + len(stage2['history']['val_pos_rmse_km'])), stage2['history']['val_pos_rmse_km'], label='stage2 val position RMSE [km]')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('RMSE')
plt.title('PINN validation position RMSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

if stage2 is not None:
    plt.figure(figsize=(10, 3))
    nbody_raw = np.array(stage2['history']['nbody_loss'])
    nbody_w = np.array(stage2['history']['nbody_weight'])
    phys_raw = np.array(stage2['history']['physics_loss'])
    phys_w = np.array(stage2['history'].get('physics_weight', [0.0] * len(nbody_raw)))
    plt.plot(nbody_raw, label='stage2 nbody raw')
    plt.plot(np.maximum(1e-16, nbody_raw * nbody_w), label='stage2 nbody weighted')
    plt.plot(phys_raw, label='stage2 phys raw')
    plt.plot(np.maximum(1e-16, phys_raw * phys_w), label='stage2 phys weighted')
    plt.yscale('log')
    plt.xlabel('epoch')
    plt.ylabel('loss contribution scale')
    plt.title('Stage2 physics contribution diagnostics')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()









Dataset source: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/PINN/data/de440.bsp
States shape: (58441, 10, 6)
Num samples: 58441


Training:   0%|          | 2/1200 [01:33<15:31:18, 46.64s/it, lr=2.00e-04, nbody=0.000e+00, nbody_w=0.00e+00, phys=0.000e+00, phys_w=0.00e+00, pos_rmse_km=3.398e+08, smooth=0.000e+00, train=2.902e+00, val=1.877e+00]


KeyboardInterrupt: 